In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from numpy.ma.core import less_equal
from tqdm import tqdm
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [2]:
# Create Model
m = gp.Model("Model_1")

Set parameter Username
Set parameter LicenseID to value 2841584
Academic license - for non-commercial use only - expires 2027-07-06


In [3]:
# Time Based GMCNF parameters

# Gravitational acceleration [m/sˆ2]
g_0 = 9.80665

# Number of nodes i,j
nodes = 4

Connections = {0: [0,1] ,
               1: [0,1,2],
               2: [1,2,3],
               3: [2,3]}

# Time Steps 12 (days) #testing with +1 day
T = 12

# Advanced Time window
T_adv = list(range(T))
print(T_adv)
#in case multiple time windows are needed this can be added


#Node Open Windows:
#Arcs can only depart or arrive at these nodes at the times specified (So always include the start and end of the window in the nodes)
#This makes every single arc unique based on its departing time and arrival node
#So: holding arc become multipliers for whatever time can be kept
N_Window = {0: [0,4,8,9,10,11],
             1:[0,5,9,10,11],
               2: T_adv,
                3:[0,2,3,4,5,6,11]}



# Velocity change [km/StructureMass]
#this model optimizes for IMLEO,
#  so it does not consider any velocity chage necessary for LEO-PAC (splashdown)
#However, since the model includes splash down, and we want only a single launch per day,
#We can add a Large (big PropCapacity ) cost for PAC to LEO to ensure that the model does not use this arc
# unless it is really necessary

#Pacific Ocean, Low Earth Orbit, Lunar Lunar Orbit, Lunar Surface
# PAC, LEO, LLO, LS are 0, 1, 2, 3
delta_V = {0: {0: 0, 1: 1000}, # PAC to LEO is Big PropCapacity high
            1: {0: 0, 1: 0, 2: 4.04},
              2: {1: 4.04, 2: 0, 3: 1.87},
                3: {2: 1.87, 3: 0}}

# Time of travel [days]
TOF = {0: {0: 1, 1: 1},
        1: {0: 1, 1: 1, 2: 3},
          2: {1: 3, 2: 1, 3: 1},
            3: {2: 1, 3: 1}}

# I, J = nodes, nodes
# For a general model, the arc routes must be manually defined, since you can only get to certain locations from certain arcs
routes = {i: [j for j in range(nodes) if abs(i - j) <= 1] for i in range(nodes)}

print(routes)

#Shows off all possible arcs
#In order [starttime][startnode][endnode]{"ArrivalTime", "FullTravelTime"}
def AllpossibleOutflowArcs(Connections, T_adv, N_Window, TOF = TOF):
  

  AllArcs = {}

  for t in T_adv:
    TimeNode = {}
    for i in Connections:
      if t in N_Window[i]:
        
        Now = N_Window[i].index(t)
        TimeNode[i] = {}
        
        for j in Connections[i]:
            
            if t+TOF[i][j] in N_Window[j]:
              TimeNode[i][j]= {"ArrivalTime": t+TOF[i][j], "FullTravelTime":TOF[i][j] }

        #If the holding arc is not available, then the arc to the next available
        #free time block is added, as long as we are not at the end of the window (N_Window[i])
        if (i not in TimeNode[i]) and (Now < len(N_Window)): 
          TimeNode[i][i]={"ArrivalTime":N_Window[i][Now+1],"FullTravelTime":N_Window[i][Now+1] - t}
        
        
    if TimeNode != {}:
      AllArcs[t] = TimeNode
            
                    
    
  return AllArcs
AllArcs = AllpossibleOutflowArcs(Connections,T_adv,N_Window=N_Window,TOF = TOF)
                
print(Connections)    
print(len(AllArcs))    
                
        
        


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
{0: [0, 1], 1: [0, 1, 2], 2: [1, 2, 3], 3: [2, 3]}
{0: [0, 1], 1: [0, 1, 2], 2: [1, 2, 3], 3: [2, 3]}
12


In [4]:

#Simplified test data


# Number of vehicle types
V = 4



Y = GRB.INTEGER
# Spacecrafts of same type


# Structure mass [kg]
StructureMass = np.array([40000, 15000,3000,255 ])

# Specific impulses [StructureMass]
I_sp = np.array([421, 324,0,0])

# Payload Capacity [kg]
PayloadCap = np.array([5000, 2500,200,12])

# Propellant Capacity [kg]
PropCapacity = np.array([1200770, 400000,0,0])


In [5]:
# VEHICLE DATA

"""






# Number of vehicle types
V = 6



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
StructureMass = np.array([38415, 12014, 4841, 6053, 2770, 1719])

# Specific impulses [StructureMass]
I_sp = np.array([421, 421, 0, 314, 311, 311])

# Payload Capacity [kg]
PayloadCap = np.array([0, 0, 524, 60, 500, 250])

# Propellant Capacity [kg] 
PropCapacity = np.array([452045, 107725, 0, 18413, 8804, 2358])




"""




'\n\n\n\n\n\n\n# Number of vehicle types\nV = 6\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]\n\n\n# Structure mass [kg]\nStructureMass = np.array([38415, 12014, 4841, 6053, 2770, 1719])\n\n# Specific impulses [StructureMass]\nI_sp = np.array([421, 421, 0, 314, 311, 311])\n\n# Payload Capacity [kg]\nPayloadCap = np.array([0, 0, 524, 60, 500, 250])\n\n# Propellant Capacity [kg] \nPropCapacity = np.array([452045, 107725, 0, 18413, 8804, 2358])\n\n\n\n\n'

In [6]:
# COMMODITY Data and Demand/Supply

# Propellant mass fraction
#defined from rocket equation 1-e**(-deltav/Ispg0)

#actually the official function is e**(-deltav/Ispg0), but using the 1-e form allows use 
# to multiply the contents with the unchanging masses to include their input into the transformation linearly
#This varies with the delta v necessary for each arc, and the Isp of the vehicle used for that arc
#Will be used later


def phi(i,j,v, dV = delta_V, I_sp = I_sp, g_0 = g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000*dV[i][j] / (I_sp[v] * g_0))) #1000 used for conversion



# Commodity variable types
# Crew, consumables kg, equipment kg, samples kg, propellant kg
X = [GRB.INTEGER, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS]

# Crew mass [kg/crew]
crew_mass = 100


CommodityMassConversion = [crew_mass,1,1,1,1] #what to multiply the commodity with to get Kg
PropIndex = 4

#In a separate list:
#Add a variable for each  rocket, listed in Carriable to then recognize the vehicle
#Carriable = {} #index of vehicle
Carriable = []
CarriedVar = [] # Variable used for the payload
for i,x in enumerate(I_sp):
    #Carriable[x] = GRB.INTEGER
    Carriable.append(i)
    CarriedVar.append(GRB.INTEGER) #add an integer variable to the commodity vector

# Crew, consumables kg, equipment kg, samples kg, propellant kg
# PAYLOAD ASSUMPTIONS

# Consumption rates [kg/crew/day]
food_consumption = 1.0
water_consumption = 5.0
oxygen_consumption = 1.1
consumption = food_consumption + water_consumption + oxygen_consumption
#if the model works, this can be made more granular by separating the consumptions




# Upper limit of each variable per spacecraft is the capacity of each spacecraft *number of spacecraft in that node.
# Crew, consumables kg, equipment kg, samples kg, propellant kg
XUpper = [PayloadCap/crew_mass, PayloadCap, PayloadCap, PayloadCap, PropCapacity]

#Single Spacecraft consumption table (from outflow to inflow consumption of all commodities)

#Matrix multiplication with a vector of outflows


# Propellant usage Matrix
def Solo_SC_Consumption(i, j,v, consumption = consumption, TOF = TOF,structure_mass = StructureMass, extraPayload = Carriable,PropellantIndex = PropIndex):


    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft

    
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [crew_mass * -1*phi(i,j,v,delta_V,I_sp,g_0),
                       -1*phi(i,j,v,delta_V,I_sp,g_0),
                         -1*phi(i,j,v,delta_V,I_sp,g_0),
                           -1*phi(i,j,v,delta_V,I_sp,g_0),
                             1-1*phi(i,j,v,delta_V,I_sp,g_0),
                                -1*structure_mass[v]*phi(i,j,v,delta_V,I_sp,g_0)], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock




    
    for i1,x1 in enumerate(Carriable):
        i2 = len(Carriable) - i1
        FullMatrix[-i2,-i2] = 1
        FullMatrix[PropellantIndex, -i2] = -1*structure_mass[x1]*phi(i,j,v,delta_V,I_sp,g_0)


    return FullMatrix


#Arc commodity transformations when there is no propellant burn
def Solo_SC_Consumption_NodV(i, j, consumption = consumption, TOF = TOF, extraPayload = Carriable):

    NumbComm = 5 #5 Commodities 
    extraNumb = len(extraPayload)

    CommodityBlocklen = NumbComm+1

    Full_Length =  NumbComm + extraNumb + 1
    FullMatrix = np.zeros((Full_Length,Full_Length))

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    CommodityBlock = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [0, 0, 0, 0, 1, 0], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    FullMatrix[:CommodityBlocklen,:CommodityBlocklen] = CommodityBlock

    for i1,x1 in enumerate(Carriable):
        i2 = len(Carriable) - i1
        FullMatrix[-i2,-i2] = 1


    return FullMatrix










# THE COMMODITIES ARE PROVIDED AT LEO. START ALL THINGS AT LEO
# CHECK DAYS FOR MISSION !!!
#Commodity Array: D[Node][Day][Commodity]



D = [[np.array([0 for x in range(len(X))])
      for _ in T_adv]
    for _ in routes]

print(len(D))
print(len(D[0]))
print(len(D[0][0]))

#initial test values
# Earth (PAC) consumables, equipment, propellant supply infinite at leo time 0, AND Moon surface sample supply (infinite at all times)
#Crew are capped in supply so we don't leave anyone on the moon
D[1][0][0] = 3 #Crew
D[1][0][1] = 99999 #Consumables
D[1][0][2] = 99999 #Equipment
D[1][0][4] = 99999999 #Propellant

for x in T_adv:
    D[3][x][3] = 999999 #Moon samples


# APOLLO
# Remember we are using a list starting at 0
# Crew demand/supply
D[3][4][0] = -2 # Lunar surface day 5 crew demand (negative supply)
D[2][3][0] = -1 # Lunar orbit day 4 crew demand
D[3][5][0] = 2 # Lunar surface day 6 crew supply (return)
D[2][6][0] = 1 # Lunar orbit day 7 crew supply (return)
D[0][10][0] = -3 # Earth day 11 crew demand (return)

D[3][4][2] = -420 # Lunar surface day 5 (scientific) equipment demand

D[0][10][3] = -110 # Earth day 11 lunar sample demand





# S/PayloadCap COMMODITY DEMAND
# format d[node][vehicle][day]
# Infinite supply of SC at LEO day 1 (time 0), none elsewhere
d = [[[1 if (i == 1 and t == 0) else 0 for t in range(T)] # Infinite supply of spacecrafts at i = 1, t = 0 LEO
     for _ in range(V)]
    for i in routes]


4
12
5


In [7]:
# CREATE COMMODITY FLOW VECTORS AND S/PayloadCap COMMODITY FLOW
# Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
#v: vehicle type, i: node of origin, j: node of destination, t: time step, x: commodity type
# Spacecraft Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'




#Only create variables for arcs that end within the destination window.

#TOF tells you the travel times
def check_destination_window(startnode, endnode, tstart, All_nodes= T_adv, TOF = TOF):
    arrival =  TOF[startnode][endnode]+tstart

    if arrival in All_nodes:
        return True
    else:
        return False


#lower bound for all commodities is 0, no negatives.
def create_commodity_flow(model, V, X, Time = T_adv, direction = "out", connect = routes, typeC = "Classic"):

    
    
    x_flow = [[{j: [np.array([[model.addVar(vtype=X[x], name=f'{typeC}_commodity_{direction}flow_{v},{i},{j},{t},{x}',lb = 0 )]
                              for x in range(len(X))])
                    for t in range(T-1) if check_destination_window(i, j, t, Time, TOF)]
                for j in connect[i]}
               for i in connect ]
              for v in range(V)]


    return x_flow


def create_sc_commodity_flow(model, V,Y, Time = T_adv, direction = "out", connect = routes):
    y_flow = [[{j: [np.array([model.addVar(vtype=Y, name=f'sc_commodity_{direction}flow_{v},{i},{j},{t}',lb=0)])
            for t in range(T-1) if check_destination_window(i, j, t, Time, TOF)]
          for j in connect[i]}
         for i in connect]
        for v in range(V)]

    return y_flow

# Outflow+ leaving from node i to j, inflow- arriving at node j from i

x_outflow, x_inflow = create_commodity_flow(m, V, X, T_adv, direction="out", connect=routes), create_commodity_flow(m, V, X, T_adv, direction="in", connect=routes)
y_outflow, y_inflow = create_sc_commodity_flow(m, V, Y, T_adv, direction="out", connect=routes), create_sc_commodity_flow(m, V, Y, T_adv, direction="in", connect=routes)






def NoSelfPayload(model,V,Flowlist, TOF = TOF,Time =T_adv, PayloadV = Carriable, connect=routes):

   

    print(PayloadV)
    # if v is a vehicle in payloadv

    for v in range(V):
        for i in connect:
            for j in connect[i]:
                for t in Time:
                    if check_destination_window(i,j,t,Time,TOF):
                        if v in PayloadV:
                            SC_index =PayloadV.index(v)

                            print(v,SC_index)

                            model.addConstr(Flowlist[v][i][j][t][SC_index][0] == 0,name=f'NoSelfPayloadConstraint_vehicle{v}_startnode{i}_endnode{j}_starttime{t}')
    

    return

SCpayload_Outflow = create_commodity_flow(m,V,CarriedVar,T_adv,"out",routes,"SCPayload")
SCpayload_Inflow =  create_commodity_flow(m,V,CarriedVar,T_adv,"in",routes,"SCPayload")

NoSelfPayload(m,V,SCpayload_Outflow,TOF,T_adv,Carriable,routes)


m.update()






[0, 1, 2, 3]
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
0 0
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
1 1
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2
2 2

In [8]:
# ADD THE CONSTRAINTS (2 & 3)
# CONSTRAINTS 2 & 3 MASS BALANCE
# Node commodity demand D vectors (positive for supply)
# sum(x[i][t]+) - sum(x[i][t]-) <= D[i][t]

import sys

#this is done for all nodes except the final day

for i in routes:
    for t in T_adv: #range(T - 1)) also works for simple tests

        
        
        
        x_outflow_sum = sum(x_outflow[v][i][j][t] 
                            if t +TOF[i][j] in T_adv
                            else np.array([[0] for _ in range(len(X))])  #packaged SC are dealt with differently
                            for v in range(V) for j in routes[i])
            #)
        # On the last day there is no outflow, the if else statement ensures
        #that only the outflows for which the spacecraft has had time to arrive are counted
    

        x_inflow_sum = sum(x_inflow[v][j][i][t - TOF[j][i]] 
                           if t - TOF[j][i] in T_adv
                           else np.array([[0] for _ in range(len(X))])
                           for v in range(V) for j in routes[i])
        # Only count the inflows for which the spacecraft has had time to arrive 
        # (or where it has had time to depart (no negative times))


        #if t == 0:
        #    print(x_outflow_sum)
        #    print(len(x_outflow_sum))
        #    print(x_inflow_sum)
        #    print(len(x_inflow_sum[0]))
        #    sys.exit()

        for x in range(len(X)):
            try:
                m.addConstr(x_outflow_sum[x][0] - x_inflow_sum[x][0] <= D[i][t][x],
                            name=f"mass_balance_x_node{i}_time{t}_comm{x}")
                #print(x_outflow_sum[x][0])
            
            except Exception as e:
                print(f"Error on node={i}, time={t}, commodity={x}: {e}")
                raise
            
            #except:
            #    print("Error on constraint for node {}, time {}, commodity {}".format(i, t, x))

        # S/PayloadCap commodity supply and demand
        for v in range(V):
            y_outflow_sum = sum(y_outflow[v][i][j][t]  
                                if t +TOF[i][j] in T_adv \
                                else np.array([0]) 
                                for j in routes[i])

            y_inflow_sum = sum(y_inflow[v][j][i][t - TOF[j][i]] 
                               if t - TOF[j][i] in T_adv \
                               else np.array([0])
                               for j in routes[i])

            
            #S/PayloadCap Payload commodity supply and demand, the payloads must be summed into the node considerations
            #so the SCpayload_Inflow and Outflow variables are summed to the exisitng SC sums. Either a ship or a payload will be added
            
            
            if  v in Carriable:
                #All payload variables are added to this sum, since the self carrying constraint is already added
                # all vehicles are payloadable, this may change
                
                SC_index =Carriable.index(v)

                payloadSumOut = sum(SCpayload_Outflow[v1][i][j][t][SC_index] 
                            if t +TOF[i][j] in T_adv
                            else np.array([0]) 
                            for v1 in range(V) 
                            for j in routes[i])
                
                y_outflow_sum = sum(y_outflow_sum, payloadSumOut)

                #if i == 1 and t == 0:
                #    print(y_outflow_sum)
                #print(payloadSumOut)
                #print(t)
                #sys.exit()
                
                payloadSumIn = sum(SCpayload_Inflow[v1][j][i][t-TOF[j][i]][SC_index] 
                            if t -TOF[j][i] in T_adv
                            else np.array([0]) 
                            for v1 in range(V) 
                            for j in routes[i])
                
                y_inflow_sum = sum(y_inflow_sum,payloadSumIn)
                


            m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t],
            name=f"SC_mass_balance_x_node{i}_time{t}_vehicle{v}")


        
        


m.update()

In [9]:
## Commodity transformation
#import sys


for i in routes:
    for j in routes[i]:
        for t in T_adv:

             if check_destination_window(i,j,t,T_adv,TOF):
                  
                for v in range(V):
                
                
                    #print(i,j,t,v)
                
                
                    Vout = np.concatenate((x_outflow[v][i][j][t],
                        np.array([y_outflow[v][i][j][t]])), axis=0)


                    Vin = np.concatenate((x_inflow[v][i][j][t],
                         np.array([y_inflow[v][i][j][t]])), axis=0)
                    




                    #print(Vin)
                    #sys.exit()


                    #create the correct consumption matrix, based on deltav and travel time
                    if delta_V[i][j] <= 0: #If there is no Delta v: there is no propellant consumption
                        Consumed =Solo_SC_Consumption_NodV(i,j,consumption,TOF,Carriable)
                        #Matrix includes all payloads
                            
                    


                    #PropIndex, tells us which commodity is the propellant
                    
                    else:
                        Consumed =Solo_SC_Consumption(i,j,v,consumption,TOF,StructureMass,Carriable)
                        
                        #for each payload only the structural mass of the carried vehicle is be added, since having payload (or propellant) is the same mass and can be placed in the Carrying storage
                        #moving propellant from 1 payload SC to a working SC will be handled at nodes, and the relevant conversion constraints are also handled there
                       

                    
                    #add SC payload variables to Vin and Vout
                    for c1 in Carriable:
                        Vin = np.append(Vin,SCpayload_Inflow[v][i][j][t][c1])
                        Vout = np.append(Vout,SCpayload_Outflow[v][i][j][t][c1])
                        
                    
                    
                    #print(Consumed)
                    #print(Vin)
                    #print(Vout)
                    #sys.exit()
                    
                    

                    


                    transformed = np.dot(Consumed,Vout)
                
                
                    for i1,(enterarc,leavearc) in enumerate(zip(transformed, Vin)):
                        
                        #print(type(enterarc))
                        #print(type(leavearc))
                        m.addConstr(enterarc == leavearc,name=f'Arc_transformationConstraint_Start{i}_End{j}_Starttime{t}_Vehicle{v}_Commodity{i1}')

                    

                    
                    #one extra constraint is needed per arc, to make sure that only the fuel in the moving SC tank is used up 
                    # Since inflow follows from outflow, it can never be a difference larger than the capacity of the tank, even if the original
                    #amount of propellant is higher due to carried SC
                    #print(x_outflow[v][i][j][t][PropIndex])
                    m.addConstr(x_inflow[v][i][j][t][PropIndex][0] >= x_outflow[v][i][j][t][PropIndex][0] - PropCapacity[v])


In [10]:
# CONSTRAINTS 5 CONCURRENCY LIMITS

# Concurrency constraint matrix
# H[x+] <= e * y+ --> Payload mass and fuel in Spacecraft does not exceed maximum capacities

#Here 3 constraints must be added
# fuel in a Spaceship <= Max Propellant Occupancy + Extra space in payload if a SC is carried (scpayload *Capacity)
#Payload (including structural mass of carried spacecraft) <= Max payload of SC
# Fuel+ payload <= Max fuel +Max payload (so:extra fuel is getting carried, but its in a tank in the payload section, so it is considered part of the payload)

import copy


def create_concurrency_constraint(connect = routes, PropellantCommodityIndex = PropIndex,
                                   MassConversion = CommodityMassConversion,
                                     SCstructMass =StructureMass, payloadSC = Carriable): # Same for all vehicles, max payload mass
    
    #There are 3 separate capacities to keep in mind: Payload, Propellant, and Payload + Prop so 3 separate rows are made 1 for each
    Payloadrow = copy.deepcopy(MassConversion) 
    Payloadrow[PropellantCommodityIndex] = 0 #Massconversion is used for all classic commodities, then the propellant is removed

    Proprow = [0] * len(MassConversion)
    Proprow[PropellantCommodityIndex] = 1

    Combinedrow = copy.deepcopy(MassConversion)

    for i1,v1 in enumerate(payloadSC):
        Payloadrow.append(StructureMass[v1])
        Proprow.append(0)
        Combinedrow.append(StructureMass[v1])

    H = [{j: np.array([Payloadrow, #payload
                        Proprow,
                        Combinedrow]) #Propellant
           for j in connect[i]}
          for i in connect]
    return H


def create_sc_design_parameters(V, PayloadCap, PropCapacity,payloadSC =Carriable):
    e = np.zeros((V,3,1+len(payloadSC)))

    for i1, e1 in enumerate(e): #First 3 
            e[i1][0][0] = PayloadCap[i1]
            e[i1][1][0] = PropCapacity[i1]
            e[i1][2][0] = PayloadCap[i1] +PropCapacity[i1]


            for i2,c1 in enumerate(Carriable): #additional values for the payload variables
                 e[i1][1][1+i2] = PropCapacity[c1] 

    #e = [np.array([[PayloadCap[v]],
    #                [PropCapacity[v]],
    #                [PayloadCap[v]+PropCapacity[v]]]) for v in range(V)]
    
    

    return e


H = create_concurrency_constraint(routes, PropIndex,CommodityMassConversion,StructureMass,Carriable)
e = create_sc_design_parameters(V, PayloadCap, PropCapacity,Carriable)
print(e)
print(len(e))


[[[5.00000e+03 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]
  [1.20077e+06 1.20077e+06 4.00000e+05 0.00000e+00 0.00000e+00]
  [1.20577e+06 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]]

 [[2.50000e+03 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]
  [4.00000e+05 1.20077e+06 4.00000e+05 0.00000e+00 0.00000e+00]
  [4.02500e+05 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]]

 [[2.00000e+02 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]
  [0.00000e+00 1.20077e+06 4.00000e+05 0.00000e+00 0.00000e+00]
  [2.00000e+02 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]]

 [[1.20000e+01 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]
  [0.00000e+00 1.20077e+06 4.00000e+05 0.00000e+00 0.00000e+00]
  [1.20000e+01 0.00000e+00 0.00000e+00 0.00000e+00 0.00000e+00]]]
4


In [11]:
# ADD THE CONSTRAINTS (5)

import sys

for v in range(V):
    for i in routes:
        for j in routes[i]:
            for t in range(T-1):
                if check_destination_window(i,j,t,T_adv,TOF):
                    
                    Extendedcommodity = x_outflow[v][i][j][t]
                    
                    #Extendedconstraint = np.zeros(len(Carriable)+1)
                    
                    Extendedconstraint = [y_outflow[v][i][j][t]]


                    for i1,c1 in enumerate(Carriable):
                        Extendedcommodity = np.append(Extendedcommodity,SCpayload_Outflow[v][i][j][t][c1])
                        Extendedconstraint.append(SCpayload_Outflow[v][i][j][t][c1])
                    
                    

                    for i1, (commodity, constraint) in enumerate(zip(np.dot(H[i][j],Extendedcommodity),
                                                     np.dot(e[v],Extendedconstraint))):

                        #print(commodity)
                        #print(constraint)
                        
                        m.addConstr(commodity <= constraint[0],name = f'Max_concurrency_constraint_row{i1}_vehicle{v}_startnode{i}_endnode{j}_starttime{t}')
                    #sys.exit()

m.update()


In [12]:
# CONSTRAINTS 6 TIME-WINDOW
# ADD THE CONSTRAINTS (6)

#Minimum value of 0 for all arcs, not time window

for v in range(V):
    for i in routes:
        for j in routes[i]:
            for t in range(T-1):
                if check_destination_window(i,j,t,T_adv,TOF):

                    for commodity_out in x_outflow[v][i][j][t]:
                        m.addConstr(commodity_out[0] >= 0)

                    for commodity_in in x_inflow[v][i][j][t]:
                        m.addConstr(commodity_in[0] >= 0)

                    m.addConstr(y_outflow[v][i][j][t][0] >= 0)
                    m.addConstr(y_inflow[v][i][j][t][0] >= 0)

m.update()

# StructureMass[v] >= 0


In [13]:
# CONSTRAINTS 7 SPACE-CRAFT MASS

In [14]:
# COST FUNCTION - INITIAL MASS AT LEO
# sum(cost * x + cost_y * StructureMass * y + coststructure mass of payload SC *payloadsc)

# x = Crew, consumables, equipment, samples, propellant, crew(return)

#Currently the model assumes we are starting at the LEO node t = 0, i=1

def create_commodity_cost(V, connect = routes, crew_mass= crew_mass):
    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1]]) if (t == 0 and i == 1)
                         else np.array([[0] for _ in range(len(X))])
                         for t in range(T-1)]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 if (t == 0 and i == 1)
                         else 0
                         for t in range(T-1)]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff

cost_coeff, sc_cost_coeff = create_commodity_cost(V, routes, crew_mass)

In [15]:
# DEFINE THE COST FUNCTION (1)

"""
#General version
cost = sum(
    np.dot(cost_coeff[v][i][j][t].T, x_outflow[v][i][j][t]) + sc_cost_coeff[v][i][j][t] * StructureMass[v] * y_outflow[v][i][j][t][0]
    for v in range(V)
    for i in routes
    for j in routes[i]
    for t in range(3)
)
"""
#specific to apollo version
#only looking at node 1 at time t = 0
cost = sum(
    np.dot(cost_coeff[v][1][j][0].T, x_outflow[v][1][j][0]) + sc_cost_coeff[v][1][j][0] * StructureMass[v] * y_outflow[v][1][j][0][0]
    for v in range(V)
    for j in routes[1]
)

cost = cost[0][0]
print(cost)

m.setObjective(cost, GRB.MINIMIZE)
m.update()

100.0 Classic_commodity_outflow_0,1,0,0,0 + Classic_commodity_outflow_0,1,0,0,1 + Classic_commodity_outflow_0,1,0,0,2 + Classic_commodity_outflow_0,1,0,0,3 + Classic_commodity_outflow_0,1,0,0,4 + 40000.0 sc_commodity_outflow_0,1,0,0 + 100.0 Classic_commodity_outflow_0,1,1,0,0 + Classic_commodity_outflow_0,1,1,0,1 + Classic_commodity_outflow_0,1,1,0,2 + Classic_commodity_outflow_0,1,1,0,3 + Classic_commodity_outflow_0,1,1,0,4 + 40000.0 sc_commodity_outflow_0,1,1,0 + 100.0 Classic_commodity_outflow_0,1,2,0,0 + Classic_commodity_outflow_0,1,2,0,1 + Classic_commodity_outflow_0,1,2,0,2 + Classic_commodity_outflow_0,1,2,0,3 + Classic_commodity_outflow_0,1,2,0,4 + 40000.0 sc_commodity_outflow_0,1,2,0 + 100.0 Classic_commodity_outflow_1,1,0,0,0 + Classic_commodity_outflow_1,1,0,0,1 + Classic_commodity_outflow_1,1,0,0,2 + Classic_commodity_outflow_1,1,0,0,3 + Classic_commodity_outflow_1,1,0,0,4 + 15000.0 sc_commodity_outflow_1,1,0,0 + 100.0 Classic_commodity_outflow_1,1,1,0,0 + Classic_commodit

In [16]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 11880 rows, 8480 columns and 34996 nonzeros (Min)
Model fingerprint: 0xd54e85c5
Model has 72 linear objective coefficients
Variable types: 3392 continuous, 5088 integer (0 binary)
Coefficient statistics:
  Matrix range     [3e-01, 1e+06]
  Objective range  [1e+00, 4e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+08]

Presolve removed 11105 rows and 7409 columns
Presolve time: 0.16s
Presolved: 775 rows, 1071 columns, 4563 nonzeros
Variable types: 620 continuous, 451 integer (159 binary)
Found heuristic solution: objective 1136713.5023

Root relaxation: objective 6.493088e+04, 444 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node

In [17]:
# x = Crew, consumables, equipment, samples, propellant
#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'

results = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}

sorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))

f = open("classic_apollo_solution.txt", "w")



for final in sorted_results:
    if sorted_results[final] != 0:
        print('%StructureMass %g' % (final, sorted_results[final]))
        f.write('%StructureMass %g' % (final, sorted_results[final]))
        f.write('\n')
f.close()


ValueError: unsupported format character 'S' (0x53) at index 1

In [ ]:
# Making a graph
keylist =list(sorted_results.keys())
SC_Outflow = [None]*V


for i in range(V):
    
    veh_vars=[x for x in keylist if 'sc_commodity_outflow_'+str(i) in x and sorted_results[x] >= 0.2]
    
    SC_Outflow[i] = veh_vars

print(SC_Outflow)
for i in SC_Outflow:
    print([sorted_results[x] for x in i])
    print([x for x in i])

#max graph size  = 100x100
graphx = [None]*T
graphy = [None]*nodes
for i in range(T):
    graphx[i] = i 
for i in range(nodes):
    graphy[i] = i

colors = ['red','green','yellow','pink','blue','brown']
for count,x in enumerate(SC_Outflow):
    for y in x:

        #plot each line indiviually, basing the thickness and color of each line on the  number and type of spacecraft
        #remember to use arcs[i][j] to plot the start and end of each line
        i = int(str(y).split(",")[1])
        j = int(str(y).split(",")[2])
        t_0 = int(str(y).split(",")[-1])
        t_end = t_0 + TOF[j][i]
        wide = sorted_results[y]

        xval = [graphx[t_0],graphx[t_end]]
        yval = [graphy[i],graphy[j]]
        plt.plot(xval,yval, linewidth = wide, marker = 'o',color=colors[count] )

plt.grid(True)    
plt.show()
        



In [ ]:
# # EQUATION 7 CONSTRAINTS
#
# # Structural Fraction (fuel dependent)
# alpha = 0.045  # LOX/kerosene
#
# # Gravitational Acceleration Earth
# g_0 = 9.8  # m/s2
#
# # Upper Bound Allowed for Propellant Tank Capacity
# M_ub = 500000  # kg
#
# # Spacecraft Impulsive Burn
# t_b = 120  # StructureMass
#
#
# # Structure Mass Variable
# def create_s_star_variables(model, v=V):
#     variables = {}
#     for v in range(V):
#         variables[v] = model.addVar(vtype=GRB.CONTINUOUS, name=f'Structure_Mass_{v}')
#     return variables
#
#
# s_star = create_s_star_variables(model=m)
#
# m.update()

In [ ]:
# # CONSTRAINTS 7
#
# for v in tqdm(V):
#     m.addConstr(s_star[v] = 2.3931 * )